# Creating synthetic observations for Galaxy Populations


In [ ]:
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from synthesizer.grid import Grid
from synthpop import GalaxyPopulation
from synthpop.model import Model, Default, Constant
from unyt import yr, Myr, Msun, Gyr, unyt_quantity, Mpc, dimensionless


In [ ]:
# load grid
grid = Grid("test_grid")

# load default model
model = Default()
model = Constant()

# Define the volume of the galaxy population
volume = 1E5 * Mpc**3

In [ ]:

# Instantiate the galaxy population
galpop = GalaxyPopulation(
    minimum_stellar_mass=1E10*Msun, 
    maximum_stellar_mass=1E11*Msun, 
    volume=volume,
    model=model,
    grid=grid,
    cosmology=cosmo,
    redshift=0.0,
    random_seed=42)

print(galpop)




In [ ]:
from synthesizer.emission_models import IncidentEmission, EmergentEmission

from synthesizer.emission_models.attenuation import PowerLaw

incident = IncidentEmission(grid=grid)
print(incident)

emergent = EmergentEmission(
    grid=grid,
    dust_curve=PowerLaw(slope=-1),
    fesc=0.1,)
print(emergent)


## Spectra

Before generating other photometry we need to generate spectra.

In [ ]:
galpop.galaxies[0].stars.tau_v



In [ ]:
# galpop.generate_spectra(incident)
galpop.generate_spectra(emergent)

galpop.plot_spectra('incident')

## Photometry

To generate broadband photometry we need to provide a `synthesizer.filters.FilterSet` object. 

In [ ]:

from synthesizer.filters import UVJ

# Get a UVJ filter collection
uvj = UVJ(new_lam=grid.lam)

galpop.generate_photometry('emergent', uvj)
galpop.generate_photometry('incident', uvj)

### Luminosity functions

In [ ]:
galpop.plot_luminosity_function('incident', 'U')
galpop.plot_luminosity_function('emergent', 'U')

### Colour diagrams

Methods exist for rapidly producing colour-magnitude and colour-colour diagrams.

In [ ]:
galpop.plot_color_color_diagram('emergent', ['U','V','J'])

## Project galaxy population to earlier time

In [ ]:
galpop_z1 = galpop.project_to_earlier_epoch(redshift=3.0)

print(galpop)
print(galpop_z1)

In [ ]:

galpop.plot_stellar_mass_function()
print(galpop.age_of_the_universe)
galpop_z1.plot_stellar_mass_function()
print(galpop_z1.age_of_the_universe)
